In [2]:
import os
import random
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from collections import defaultdict
from transformers import AutoTokenizer, AutoModel, AdamW
from tqdm import tqdm

# =============================================================================
# [1] Hyperparameter Configuration
# =============================================================================
# BERT-Base 모델을 사용하여 문맥 이해도를 높이고, Self-Training 루프를 통해
# Labeled Data의 부족함을 Unlabeled Data(Pseudo-labeling)로 보완하는 설정입니다.
CONFIG = {
    'model_name': 'bert-base-uncased',   # 문맥 파악 능력이 우수한 Base 모델 채택
    'batch_size': 32,                    # VRAM 효율성과 학습 안정성을 고려한 배치 크기
    'max_len': 128,                      # 리뷰의 핵심 정보 손실을 최소화하기 위한 시퀀스 길이 확장 (64 -> 128)
    'lr': 2e-5,                          # Fine-tuning 시 Catastrophic Forgetting 방지를 위한 저수준 학습률
    'epochs_init': 3,                    # 초기 Clean Data에 대한 충분한 학습 (Underfitting 방지)
    'epochs_st': 2,                      # Self-training 단계에서의 추가 학습 에폭
    'st_iterations': 3,                  # Pseudo-labeling 반복 횟수 (점진적 라벨 정제)
    'confidence_threshold': 0.8,         # Noise Label 생성을 억제하기 위한 높은 신뢰도 임계값
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

def set_seed(seed):
    """
    실험의 재현성(Reproducibility)을 보장하기 위해 
    CPU, GPU, Numpy, Python Random의 시드를 고정합니다.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])
print(f"Running on device: {CONFIG['device']}")

# =============================================================================
# [2] File Paths
# =============================================================================
ROOT = Path("Amazon_products")

TRAIN_CORPUS_PATH = ROOT / "train" / "train_corpus.txt"
TEST_CORPUS_PATH  = ROOT / "test" / "test_corpus.txt"
CLASSES_PATH      = ROOT / "classes.txt"
HIERARCHY_PATH    = ROOT / "class_hierarchy.txt"
KEYWORDS_PATH     = ROOT / "class_related_keywords.txt"
SUBMISSION_DIR    = ROOT / "submission"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# [3] Data Loading & Preprocessing Utilities
# =============================================================================
def load_classes(path):
    id2name, name2id = {}, {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                cid, cname = int(parts[0]), parts[1]
                id2name[cid] = cname
                name2id[cname] = cid
    return id2name, name2id

def load_hierarchy(path):
    """
    계층적 분류(Hierarchical Classification) 구조를 반영하기 위해 
    부모-자식 노드 관계를 로드합니다. (Multi-label 확장에 활용)
    """
    parents = defaultdict(list)
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                p, c = int(parts[0]), int(parts[1])
                parents[c].append(p)
    return parents

def load_keywords(path, name2id):
    class_keywords = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if ':' in line:
                cname, kw_str = line.strip().split(':', 1)
                keywords = [k.strip() for k in kw_str.split(',')]
                if cname in name2id:
                    class_keywords[name2id[cname]] = keywords
    return class_keywords

def load_corpus(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t', 1)
            if len(parts) == 2:
                did, text = parts
                data.append((int(did), text))
    data.sort(key=lambda x: x[0])
    return data

# =============================================================================
# [4] Distant Supervision: Heuristic Label Generation
# =============================================================================
def generate_silver_labels_strict(corpus, class_keywords, parents_map, num_classes):
    """
    [Weak Supervision Strategy]
    초기 학습 데이터가 없는 상황에서 키워드 매칭을 통해 'Silver Label(Noise가 섞인 라벨)'을 생성합니다.
    단순 문자열 매칭(in) 대신 정규표현식(\b)을 사용하여 Precision을 높여, 
    Self-Training의 초기 Seed 데이터 품질을 확보하는 것이 핵심입니다.
    """
    print("Generating Silver Labels with Regex Boundary Matching...")
    labeled_data = [] 
    unlabeled_data = [] 
    
    # O(N*M) 매칭 속도 개선을 위해 정규표현식 객체를 미리 컴파일(Pre-compile)
    compiled_keywords = {}
    for cid, kws in class_keywords.items():
        # '\b'를 사용하여 'apple'이 'pineapple'에 매칭되는 False Positive 방지
        pattern_str = r'\b(' + '|'.join(map(re.escape, kws)) + r')\b'
        compiled_keywords[cid] = re.compile(pattern_str)

    for did, text in tqdm(corpus, desc="Heuristic Labeling"):
        text_lower = text.lower()
        matched_classes = set()
        
        # 1. Exact Keyword Matching
        for cid, pattern in compiled_keywords.items():
            if pattern.search(text_lower):
                matched_classes.add(cid)
        
        # 2. Hierarchy Propagation (자식 클래스 -> 부모 클래스 라벨 전파)
        if matched_classes:
            queue = list(matched_classes)
            visited = set(queue)
            while queue:
                curr = queue.pop(0)
                if curr in parents_map:
                    for p in parents_map[curr]:
                        if p not in visited:
                            visited.add(p)
                            queue.append(p)
            final_labels = list(visited)
            
            # Multi-hot Encoding 벡터 생성
            label_vec = np.zeros(num_classes, dtype=np.float32)
            for cid in final_labels:
                label_vec[cid] = 1.0
            labeled_data.append((text, label_vec))
        else:
            # 키워드가 없는 데이터는 Unlabeled로 분류하여 추후 Self-Training 대상으로 활용
            unlabeled_data.append((text, np.zeros(num_classes, dtype=np.float32)))
            
    print(f"Labeled (Seed): {len(labeled_data)} / Unlabeled (Target): {len(unlabeled_data)}")
    return labeled_data, unlabeled_data

# =============================================================================
# [5] Model Architecture & Dataset Definition
# =============================================================================
class TaxoClassModel(nn.Module):
    def __init__(self, model_name, num_classes):
        super(TaxoClassModel, self).__init__()
        # Pre-trained BERT를 백본으로 사용 (Transfer Learning)
        self.bert = AutoModel.from_pretrained(model_name)
        
        # Multi-label Classification을 위한 Linear Head
        # BERT의 Hidden Size(768) -> Class 개수
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)
        self.dropout = nn.Dropout(0.1) # Overfitting 방지

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # [CLS] 토큰의 벡터(Context Vector)를 추출하여 분류에 사용
        cls_token = outputs[0][:, 0, :]
        logits = self.classifier(self.dropout(cls_token))
        return logits

class BertDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        text, labels = self.data[idx]
        # Dynamic Padding 대신 배치 단위 처리를 위해 Max Len으로 고정 (속도 최적화 대신 안정성 선택)
        encoding = self.tokenizer.encode_plus(
            str(text),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(labels, dtype=torch.float)
        }

# =============================================================================
# [6] Main Execution Pipeline
# =============================================================================
def run():
    # 1. Resource Loading
    print("Loading Taxonomy Resources...")
    id2name, name2id = load_classes(CLASSES_PATH)
    parents_map = load_hierarchy(HIERARCHY_PATH)
    class_keywords = load_keywords(KEYWORDS_PATH, name2id)
    
    train_raw = load_corpus(TRAIN_CORPUS_PATH)
    test_raw = load_corpus(TEST_CORPUS_PATH)
    
    num_classes = len(id2name)
    
    # 2. Generate Initial Seed Data (Silver Labels)
    labeled_train, unlabeled_train = generate_silver_labels_strict(train_raw, class_keywords, parents_map, num_classes)
    
    # 3. Model Initialization
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
    model = TaxoClassModel(CONFIG['model_name'], num_classes).to(CONFIG['device'])
    
    # Optimizer: AdamW 사용 (Weight Decay를 통해 일반화 성능 향상)
    optimizer = AdamW(model.parameters(), lr=CONFIG['lr'])
    
    # Loss Function: Multi-label 분류이므로 BCEWithLogitsLoss 사용 (Sigmoid 내장)
    criterion = nn.BCEWithLogitsLoss()
    
    # 4. Phase 1: Supervised Fine-tuning (SFT)
    # 키워드 매칭으로 생성된 High-Precision 데이터로 모델의 초기 결정 경계를 형성
    print(f"\n=== Phase 1: Initial Training ({CONFIG['epochs_init']} Epochs) ===")
    labeled_dataset = BertDataset(labeled_train, tokenizer, CONFIG['max_len'])
    train_loader = DataLoader(labeled_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
    
    model.train()
    for epoch in range(CONFIG['epochs_init']):
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            mask = batch['attention_mask'].to(CONFIG['device'])
            labels = batch['labels'].to(CONFIG['device'])
            
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask=mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

    # 5. Phase 2: Self-Training (Semi-supervised Learning)
    # [Teacher-Student 구조]
    # 학습된 모델(Teacher)이 Unlabeled 데이터에 대해 Pseudo-label을 생성하고,
    # 신뢰도(Confidence)가 높은 데이터만 선별하여 다시 학습(Student)에 활용
    
    # Test 데이터도 Transductive Learning 관점에서 Unlabeled 데이터로 활용 가능
    candidate_data = unlabeled_train + [(t, np.zeros(num_classes)) for _, t in test_raw]
    
    for loop in range(CONFIG['st_iterations']):
        print(f"\n=== Phase 2: Self-Training Iteration {loop+1}/{CONFIG['st_iterations']} ===")
        
        # Step A: Inference on Unlabeled Data (Pseudo-labeling)
        candidate_loader = DataLoader(BertDataset(candidate_data, tokenizer, CONFIG['max_len']), 
                                      batch_size=CONFIG['batch_size']*2, shuffle=False)
        model.eval()
        new_samples = []
        
        with torch.no_grad():
            for i, batch in enumerate(tqdm(candidate_loader, desc="Generating Pseudo-labels")):
                input_ids = batch['input_ids'].to(CONFIG['device'])
                mask = batch['attention_mask'].to(CONFIG['device'])
                logits = model(input_ids, attention_mask=mask)
                probs = torch.sigmoid(logits).cpu().numpy()
                
                start_idx = i * (CONFIG['batch_size']*2)
                for j, p_vec in enumerate(probs):
                    # Step B: Confidence Thresholding (Noise Filtering)
                    # 예측 확률이 특정 임계값(0.8) 이상인 경우에만 학습 데이터로 편입하여
                    # Confirmation Bias(잘못된 라벨을 진실로 믿고 학습하는 현상)를 최소화
                    top_k = np.argsort(p_vec)[-3:]
                    if np.mean(p_vec[top_k]) >= CONFIG['confidence_threshold']:
                        lbl = np.zeros(num_classes, dtype=np.float32)
                        lbl[top_k] = 1.0 # Soft label 대신 Hard label 적용
                        new_samples.append((candidate_data[start_idx+j][0], lbl))
        
        print(f"Pseudo-labels added: {len(new_samples)} samples")
        if not new_samples: break
            
        # Step C: Re-training on Augmented Dataset
        current_train = labeled_train + new_samples
        train_loader = DataLoader(BertDataset(current_train, tokenizer, CONFIG['max_len']), 
                                  batch_size=CONFIG['batch_size'], shuffle=True)
        model.train()
        for epoch in range(CONFIG['epochs_st']):
            for batch in tqdm(train_loader, desc=f"Retraining (ST Epoch {epoch+1})"):
                input_ids = batch['input_ids'].to(CONFIG['device'])
                mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                
                optimizer.zero_grad()
                logits = model(input_ids, attention_mask=mask)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()

    # 6. Final Inference & Submission
    print("\n=== Final Inference ===")
    test_ids = [did for did, _ in test_raw]
    test_texts = [t for _, t in test_raw]
    test_data_for_loader = [(t, np.zeros(num_classes)) for t in test_texts]
    
    test_loader = DataLoader(BertDataset(test_data_for_loader, tokenizer, CONFIG['max_len']), 
                             batch_size=CONFIG['batch_size']*2, shuffle=False)
    
    model.eval()
    predicted_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            mask = batch['attention_mask'].to(CONFIG['device'])
            logits = model(input_ids, attention_mask=mask)
            probs = torch.sigmoid(logits)
            
            # Top-3 Prediction Strategy
            _, top_k_indices = torch.topk(probs, k=3, dim=1)
            predicted_labels.extend(top_k_indices.cpu().numpy())
            
    # Save Results
    SUBMISSION_PATH = SUBMISSION_DIR / "submission.csv"
    submission = pd.DataFrame({
        'id': test_ids,
        'label': [','.join(str(l) for l in labels) for labels in predicted_labels] 
    })

    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"Submission saved successfully to: {SUBMISSION_PATH}")

if __name__ == "__main__":
    run()

Running on device: cuda
Loading Data...
Generating Silver Labels (Regex Optimized)...


Labeling: 100%|██████████| 29487/29487 [02:44<00:00, 178.81it/s]


Labeled: 19500 / Unlabeled: 9987


/opt/conda/envs/esci/lib/python3.9/site-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(



=== Phase 1: Initial Training ===


...

Added 8626 pseudo-labels.


Re-training: 100%|██████████| 879/879 [04:09<00:00,  3.53it/s]



=== Final Inference ===


100%|██████████| 308/308 [01:04<00:00,  4.78it/s]

Submission saved to: Amazon_products/submission/submission.csv
